In [182]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [183]:
# import sys, os
# sys.path.append(os.path.abspath(".."))
# from src.utils import OPTIONS, clean, map_at_3, make_submission

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import wandb

In [184]:
import re

OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text, is_prompt=True):
    return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

In [185]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [186]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [187]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bert = AutoModel.from_pretrained(MODEL_NAME)
bert.to(DEVICE)
bert.eval()

In [189]:
@torch.no_grad()
def embed_texts(texts, batch_size=32):
    vectors = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = bert(**encoded)
        cls = outputs.last_hidden_state[:,0]
        cls = F.normalize(cls, dim=1)
        vectors.append(cls.cpu())

    return torch.cat(vectors).numpy()

In [190]:
train['question'] = train['prompt'].map(clean)
test['question'] = test['prompt'].map(clean)

for opt in OPTIONS:
    train[f'{opt}_clean'] = train[opt].map(lambda x : clean(x, is_prompt=False))
    test[f'{opt}_clean'] = test[opt].map(lambda x : clean(x, is_prompt=False))

In [191]:
def combined_format(df, has_labels=True):
    rows = []
    for _, row in df.iterrows():
        for opt in OPTIONS:
            rows.append({
                'id': row['id'],
                'option': opt,
                'combined': row['question'] + ' [SEP] ' + row[f'{opt}_clean'],
                'label': int(row['answer'] == opt) if has_labels else None
            })
    return pd.DataFrame(rows)
 
train_tf = combined_format(train, has_labels=True)
test_tf = combined_format(test, has_labels=False)
 
print(train_tf.head(5))
train_tf['combined'][0]

   id option                                           combined  label
0   1      A  what is martin heidegger s view on the relatio...      0
1   1      B  what is martin heidegger s view on the relatio...      1
2   1      C  what is martin heidegger s view on the relatio...      0
3   1      D  what is martin heidegger s view on the relatio...      0
4   1      E  what is martin heidegger s view on the relatio...      0


'what is martin heidegger s view on the relationship between time and human existence [SEP] martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond one s own time'

In [192]:
bert.eval()
X = embed_texts(train_tf['combined'].tolist())
y = train_tf.label.values

In [193]:
groups = train_tf['id'].values
gkf = GroupKFold(n_splits=5)
 
fold_scores = []
 
model = LogisticRegression(max_iter=3000, C=3, class_weight={0:1, 1:4}) 

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    model.fit(X[tr_idx], y[tr_idx])
 
    val = train_tf.iloc[val_idx].copy()
    val['prob'] = model.predict_proba(X[val_idx])[:, 1]

    preds, truths = [], []
    for qid, group in val.groupby('id'):
        ranked = group.sort_values('prob', ascending=False)['option'].tolist()
        preds.append(ranked[:3])
        truths.append(train.loc[train['id'] == qid, 'answer'].values[0])
 
    score = map_at_3(truths, preds)
    fold_scores.append(score)
    print(f"Fold{fold} MAP@3: {score:.4f}")

Fold0 MAP@3: 0.7925
Fold1 MAP@3: 0.7433
Fold2 MAP@3: 0.7617
Fold3 MAP@3: 0.7475
Fold4 MAP@3: 0.7392


In [194]:
model.fit(X, y)

LogisticRegression(C=3, class_weight={0: 1, 1: 4}, max_iter=3000)

In [195]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

WANDB = user_secrets.get_secret("WAND")

In [196]:
wandb.login(key=WANDB)

wandb.init(
    project="23f2003391-t22026",
    name="bert-base-lr-cls-pooling",
    config={
        "model": "bert-base-uncased",
        "pooling": "cls",
        "classifier": "LogisticRegression",
        "C": 3,
        "class_weight": "{0:1, 1:4}",
        "max_length": 512,
        "n_folds": 5,
    }
)

for i, score in enumerate(fold_scores):
    wandb.log({"fold": i, "fold_map3": score})

wandb.log({
    "mean_map3": np.mean(fold_scores),
    "std_map3": np.std(fold_scores)
})

wandb.finish()
print(f"Mean MAP@3: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


fold,▁▃▅▆█
fold_map3,█▂▄▂▁
mean_map3,▁
std_map3,▁
fold,4
fold_map3,0.73917
mean_map3,0.75683
std_map3,0.01937


Mean MAP@3: 0.7568 ± 0.0194


In [197]:
X_test = embed_texts(test_tf['combined'].tolist())
test_tf['prob'] = model.predict_proba(X_test)[:, 1]
 
test_preds = []
test_ids = []
for qid, group in test_tf.groupby('id'):
    ranked = group.sort_values('prob', ascending=False)['option'].tolist()
    test_preds.append(ranked[:3])
    test_ids.append(qid)
 
submission = make_submission(test_ids, test_preds)
submission.to_csv('submission.csv', index=False)
submission.head()

,id,prediction
0,1,C E B
1,2,B E C
2,3,B E D
3,4,D B E
4,5,B C A
